# Med-Guard: Benchmark — Per-sample Latency & Memory

This notebook benchmarks the `MedGuardInference` per-sample latency, throughput, and memory usage using synthetic signals. It performs warm-up runs, per-sample timing, a throughput test, a memory snapshot via `tracemalloc`, and a parameter sweep over `window_size`.


# Setup / Optional dependency install (uncomment to run)
# !pip install -r requirements.txt
# !pip install numpy pandas matplotlib

# Notes: On Windows use the commands in a terminal; these are optional when running in a pre-configured env.

In [ ]:
# 1) Imports and configuration
import sys
import os
import time
import timeit
import tracemalloc
import json
import csv
import statistics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook config
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

np.random.seed(42)
import random
random.seed(42)

# Ensure local src is on path
ROOT = os.path.abspath(os.path.join(os.getcwd()))
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print('Python', sys.version.splitlines()[0])
print('NumPy', np.__version__)
print('Pandas', pd.__version__)
print('Notebook root:', ROOT)


In [ ]:
# 2) Load MedGuardInference and quick sanity check
try:
    from inference import MedGuardInference
except Exception as e:
    # Fallback: try package-style import if available
    try:
        from src.inference import MedGuardInference
    except Exception:
        raise

engine = MedGuardInference(window_size=15, threshold=2.0)
print('Engine instantiated:', type(engine).__name__)

# Basic API check
sample = 80
res = engine.analyze(sample)
print('analyze({}) -> {}'.format(sample, res))


In [ ]:
# 3) Synthetic signal generator helper
import math

def synthetic_signal(length=1000, sampling_rate=1.0, noise_std=1.0, anomaly_indices=None, amplitude=10.0):
    t = np.arange(length) / sampling_rate
    # baseline: low-frequency sinusoid
    baseline = 60 + amplitude * np.sin(2 * math.pi * 0.01 * t)
    noise = np.random.normal(loc=0.0, scale=noise_std, size=length)
    signal = baseline + noise
    # step changes
    if length > 200:
        signal[200:220] += 5.0
    if anomaly_indices:
        for idx in anomaly_indices:
            if 0 <= idx < length:
                signal[idx] += amplitude * 4.0  # large spike
    return signal

# Quick sanity
sig = synthetic_signal(50, noise_std=0.5, anomaly_indices=[10,30])
print('sample signal (first 10):', sig[:10])


In [ ]:
# 4) Functional check and z-score verification
# Populate the engine buffer with an initial window, then verify z_score for a known value
window_size = 15
engine = MedGuardInference(window_size=window_size, threshold=2.0)
init_vals = synthetic_signal(window_size, noise_std=0.5)
for v in init_vals[:-1]:
    engine.analyze(float(v))
# last value (the test sample)
test_sample = float(init_vals[-1])
manual_mu = np.mean(init_vals[:-1])
manual_sigma = np.std(init_vals[:-1], ddof=0)
manual_z = (test_sample - manual_mu) / (manual_sigma if manual_sigma>0 else 1e-9)
res = engine.analyze(test_sample)
print('manual z:', manual_z)
print('engine result:', res)
# If engine returns z_score as second element, compare numerically (best-effort)
if isinstance(res, (list, tuple)) and len(res) >= 2:
    engine_z = float(res[1])
    print('engine z:', engine_z)
    assert abs(engine_z - manual_z) < 1e-6 or np.isclose(engine_z, manual_z, atol=1e-3), 'z-score mismatch (within tolerance)'


In [ ]:
# 5) Warm-up runs (first-call bias)
warmup_runs = 200
engine = MedGuardInference(window_size=15, threshold=2.0)
data = synthetic_signal(warmup_runs + 10, noise_std=0.5)
for v in data[:warmup_runs]:
    engine.analyze(float(v))
print('Warm-up completed ({} calls)'.format(warmup_runs))


In [ ]:
# 6) Throughput benchmark (batch processing timing)
N = 5000
engine = MedGuardInference(window_size=15, threshold=2.0)
samples = synthetic_signal(N, noise_std=0.5)

start = time.perf_counter()
for v in samples:
    engine.analyze(float(v))
end = time.perf_counter()
elapsed = end - start
throughput = N / elapsed if elapsed>0 else float('inf')
print('Processed {} samples in {:.4f} s — {:.1f} samples/s'.format(N, elapsed, throughput))


In [ ]:
# 7) Per-sample latency microbenchmark (stat summary)
# Measure per-call durations and compute summary stats
M = 2000
engine = MedGuardInference(window_size=15, threshold=2.0)
samples = synthetic_signal(M, noise_std=0.5)
latencies = np.empty(M, dtype=float)

for i, v in enumerate(samples):
    t0 = time.perf_counter()
    engine.analyze(float(v))
    t1 = time.perf_counter()
    latencies[i] = (t1 - t0)

# Convert to milliseconds
lat_ms = latencies * 1000
print('Latency (ms): mean={:.4f}, median={:.4f}, p95={:.4f}, max={:.4f}'.format(
    np.mean(lat_ms), np.median(lat_ms), np.percentile(lat_ms,95), np.max(lat_ms)
))


In [ ]:
# 8) Memory profiling with tracemalloc
tracemalloc.start()
engine = MedGuardInference(window_size=15, threshold=2.0)
samples = synthetic_signal(2000, noise_std=0.5)
for v in samples:
    engine.analyze(float(v))

snapshot = tracemalloc.take_snapshot()
tracemalloc.stop()

# Report top allocations
top_stats = snapshot.statistics('lineno')[:10]
print('Top memory allocations (top 10):')
for stat in top_stats:
    print(stat)

# Current and peak memory (if available)
# Note: tracemalloc provides peak via get_traced_memory when running


In [ ]:
# 9) Parameter sweep: window_size vs latency
configs = []
window_sizes = [5, 15, 30, 60]
thresholds = [1.5, 2.0, 3.0]

for ws in window_sizes:
    for th in thresholds:
        engine = MedGuardInference(window_size=ws, threshold=th)
        samples = synthetic_signal(2000, noise_std=0.5)
        # warm-up
        for v in samples[:200]:
            engine.analyze(float(v))
        # timed run
        t0 = time.perf_counter()
        for v in samples:
            engine.analyze(float(v))
        t1 = time.perf_counter()
        elapsed = t1 - t0
        throughput = len(samples) / elapsed if elapsed>0 else float('inf')
        configs.append({'window_size': ws, 'threshold': th, 'throughput': throughput, 'elapsed_s': elapsed})
        print('ws', ws, 'th', th, '->', '{:.1f} sps'.format(throughput))

df_configs = pd.DataFrame(configs)
df_configs

In [ ]:
# 10) Aggregate results, plots, and save outputs
import os
out_dir = os.path.join(ROOT, 'notebooks', 'benchmark_outputs')
os.makedirs(out_dir, exist_ok=True)

# Throughput plot vs window_size
plt.figure(figsize=(8,4))
for th in df_configs['threshold'].unique():
    sub = df_configs[df_configs['threshold']==th]
    plt.plot(sub['window_size'], sub['throughput'], marker='o', label=f'th={th}')
plt.xlabel('window_size')
plt.ylabel('throughput (samples/sec)')
plt.title('Throughput vs window_size')
plt.legend()
plt.grid(True)
plt.tight_layout()
fig_path = os.path.join(out_dir, 'throughput_vs_window.png')
plt.savefig(fig_path)
print('Saved figure:', fig_path)

# Save numeric results
csv_path = os.path.join(out_dir, 'benchmark_results.csv')
df_configs.to_csv(csv_path, index=False)
print('Saved results CSV:', csv_path)


In [ ]:
# 11) Optional automated unit-test hook (pytest)
# This small test runs a short benchmark and asserts median latency < threshold_ms

def test_short_benchmark():
    import os
    thr_ms = float(os.environ.get('MEDGUARD_LATENCY_MS', '20'))
    engine = MedGuardInference(window_size=15, threshold=2.0)
    samples = synthetic_signal(200, noise_std=0.5)
    lat = []
    for v in samples:
        t0 = time.perf_counter()
        engine.analyze(float(v))
        t1 = time.perf_counter()
        lat.append((t1-t0)*1000.0)
    median_ms = float(np.median(lat))
    assert median_ms < thr_ms, f"Median latency {median_ms:.2f} ms >= {thr_ms} ms"

print('pytest hook added: run `pytest -q` to execute the test (env MEDGUARD_LATENCY_MS to configure)')
